# Abrir librerías

In [1]:
import os
os.chdir('..')

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier 
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import accuracy_score, recall_score, precision_score, confusion_matrix, classification_report


from scripts.load_data import load_data

# Load Data

In [3]:
n_test, train_smile, train_nonSmile, test_smile, test_nonSmile = load_data()

In [4]:
test_smile

array([[[[0.29411766, 0.26666668, 0.25490198],
         [0.29411766, 0.26666668, 0.25490198],
         [0.29411766, 0.26666668, 0.25490198],
         ...,
         [0.44705883, 0.39607844, 0.3647059 ],
         [0.44705883, 0.39607844, 0.3647059 ],
         [0.44705883, 0.39607844, 0.3647059 ]],

        [[0.29411766, 0.26666668, 0.25490198],
         [0.29411766, 0.26666668, 0.25490198],
         [0.29411766, 0.26666668, 0.25490198],
         ...,
         [0.44705883, 0.39607844, 0.3647059 ],
         [0.44705883, 0.39607844, 0.3647059 ],
         [0.44705883, 0.39607844, 0.3647059 ]],

        [[0.29411766, 0.26666668, 0.25490198],
         [0.29411766, 0.26666668, 0.25490198],
         [0.29411766, 0.26666668, 0.25490198],
         ...,
         [0.44705883, 0.39607844, 0.3647059 ],
         [0.44705883, 0.39607844, 0.3647059 ],
         [0.44705883, 0.39607844, 0.3647059 ]],

        ...,

        [[0.4509804 , 0.4509804 , 0.43529412],
         [0.4509804 , 0.4509804 , 0.43529412]

In [ ]:
#join trainning
train_features = np.concatenate((train_smile, train_nonSmile), axis=0)
train_labels = np.concatenate((np.full(len(train_smile), "smile"), 
                               np.full(len(train_nonSmile), "non-smile")))

#join test
test_features = np.concatenate((test_smile, test_nonSmile), axis=0)
test_labels = np.concatenate((np.full(len(test_smile), "smile"), 
                              np.full(len(test_nonSmile), "non-smile")))


# Shuffle train data
idxs = np.random.choice(len(train_features), size=len(train_features), replace=False)
train_features = train_features[idxs,:,:,:]
train_labels = train_labels[idxs]

# Shuffle test data
idxs = np.random.choice(len(test_features), size=len(test_features), replace=False)
test_features = test_features[idxs,:,:,:]
test_labels = test_labels[idxs]

In [6]:
# Flatten the image data (assuming images are in format [samples, height, width, channels])
x_train_flat = train_features.reshape(train_features.shape[0], -1)
x_test_flat = test_features.reshape(test_features.shape[0], -1)
y_train = train_labels
y_test = test_labels

In [7]:
x = x_train_flat
y = y_train

# Fuciones

In [8]:
# Hacer Cross validation dependiendo del modelo; se resume mucho el código ****
def evaluate_stratified_shuffle_split(X, y, modelo_kernel, train_size=0.8, n_splits=5,
                                          random_state=1234, getter=False):
    sss = StratifiedShuffleSplit(
        n_splits=n_splits,
        train_size=train_size,
        #random_state=random_state
    )
    classes = np.unique(y)
    num_classes = len(classes)
    acc = 0
    recall = np.zeros(num_classes)
    precision = np.zeros(num_classes)

    cv_y_test = []
    cv_y_pred = [] 

    for train_index, test_index in sss.split(X, y):
        X_train, X_test = X[train_index], X[test_index]
        y_train, y_test = y[train_index], y[test_index]

        clf = RandomForestClassifier(max_depth=10, random_state=1234)

        clf.fit(X_train, y_train)
        y_pred = clf.predict(X_test)
        cv_y_test.append(y_test)
        cv_y_pred.append(y_pred)
        acc += accuracy_score(y_test, y_pred)
        recall += recall_score(y_test, y_pred, average=None, labels=classes)
        precision += precision_score(y_test, y_pred, average=None, labels=classes, zero_division=0)


    acc_mean = acc / n_splits
    precision_mean = precision / n_splits
    recall_mean = recall / n_splits
    y_test_all = np.concatenate(cv_y_test)
    y_pred_all = np.concatenate(cv_y_pred)
    cm = confusion_matrix(y_test_all, y_pred_all, labels=classes)
    report = classification_report(y_test_all, y_pred_all, target_names=classes, zero_division=0)
    results = {
        'accuracy': acc_mean,
        'precision': dict(zip(classes, precision_mean)),
        'recall': dict(zip(classes, recall_mean)),
        'confusion_matrix': cm,
        'classification_report': report,
        'y_test_all': y_test_all,
        'y_pred_all': y_pred_all,
        'classes': classes
    }
    
    if getter:
        observacion = [train_size, acc_mean]
        # Agregar recall para cada clase en orden
        for cls in sorted(classes):
            observacion.append(recall_mean[np.where(classes == cls)[0][0]])
        return observacion
    else:
        print("------------------")
        print(modelo_kernel)
        print("------------------")
        print(f'Accuracy promedio: {results["accuracy"]:.4f}')
        print("------------------")
        print('Precision por clase:')
        for cls, prec in results["precision"].items():
            print(f'{cls}: {prec:.4f}')
        print("------------------")
        print('Recall por clase:')
        for cls, rec in results["recall"].items():
            print(f'{cls}: {rec:.4f}')
        print("------------------")
        print(f'Matriz de confusión:\n{results["confusion_matrix"]}')
        print("Clases:", results["classes"])
        print("------------------")
        print(f'Reporte de clasificación:\n{results["classification_report"]}')
        print("------------------")

# Simple evaluation

In [11]:
from sklearn.metrics import  f1_score, confusion_matrix, classification_report

In [ ]:
max_depth = 10
random_state = 1234


clf = RandomForestClassifier(max_depth=max_depth, random_state=random_state)
clf.fit(x, y)

y_pred = clf.predict(x_test_flat)

# ---------

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, pos_label="smile")
recall = recall_score(y_test, y_pred, pos_label="smile")
f1 = f1_score(y_test, y_pred, pos_label="smile")
cm = confusion_matrix(y_test, y_pred, labels=["smile", "non-smile"])

# First evaluation

In [9]:
# First evaluation
modelo = "Random Forest - Smile Detection"
print(f"Ejecutando modelo: {modelo}")
evaluacion = evaluate_stratified_shuffle_split(x, y, modelo_kernel=modelo, getter=False)
print(f"Fin de Ejecución del modelo: {modelo}")
print("#" * 50)  # Divisor 


Ejecutando modelo: Random Forest - Smile Detection
------------------
Random Forest - Smile Detection
------------------
Accuracy promedio: 0.8604
------------------
Precision por clase:
non-smile: 0.8317
smile: 0.8958
------------------
Recall por clase:
non-smile: 0.9042
smile: 0.8167
------------------
Matriz de confusión:
[[434  46]
 [ 88 392]]
Clases: ['non-smile' 'smile']
------------------
Reporte de clasificación:
              precision    recall  f1-score   support

   non-smile       0.83      0.90      0.87       480
       smile       0.89      0.82      0.85       480

    accuracy                           0.86       960
   macro avg       0.86      0.86      0.86       960
weighted avg       0.86      0.86      0.86       960

------------------
Fin de Ejecución del modelo: Random Forest - Smile Detection
##################################################


# Optimization of hiperparameters

in this case, max depth

In [ ]:
def optimize_max_depth(x, y, mm_range=np.arange(1, 20), n_splits=5, train_size=0.8, random_state=1234):
    acc1 = []

    for m in mm_range:
        
        acc_cv1 = []

        sss = StratifiedShuffleSplit(
            n_splits=n_splits,
            train_size=train_size,
            random_state=random_state
        ) 

        for train_index, test_index in sss.split(x, y):
        
            # Training phase
            x_train1 = x[train_index, :]
            y_train1 = y[train_index]     

            clf_cv1 = RandomForestClassifier(max_depth=m, random_state=random_state)           

            clf_cv1.fit(x_train1, y_train1)

            # Test phase
            x_test1 = x[test_index, :]
            y_test1 = y[test_index]
            y_pred1 = clf_cv1.predict(x_test1)    
            
            acc_i1 = accuracy_score(y_test1, y_pred1)
            acc_cv1.append(acc_i1)    

        acc_hyp1 = np.average(acc_cv1)
        acc1.append(acc_hyp1)

    opt_index1 = np.argmax(acc1)
    opt_hyperparameter1 = mm_range[opt_index1]
    opt_acc1 = acc1[opt_index1]

    print("Optimal max depth: ", opt_hyperparameter1)
    print("Its accuracy: ", opt_acc1)
    
    plt.figure(figsize=(10, 6))
    plt.plot(mm_range, acc1)
    plt.xlabel("Max Depth")
    plt.ylabel("Accuracy")
    plt.title("Max Depth Optimization")
    plt.show()
    
    return opt_hyperparameter1, opt_acc1

In [ ]:
opt_hyperparameter1, opt_acc1 = optimize_max_depth(x, y)


# Feature Selection

In [ ]:
n_feats = np.arange(1,86)

acc_nfeat1 = []

for n_feat in n_feats:
    
    acc_cv1 = []

    sss = StratifiedShuffleSplit(
        n_splits=5,
        train_size=0.8,
        random_state=1234
    ) 

    for train_index, test_index in sss.split(x, y):
    
        # Training phase
        x_train1 = x[train_index, :]
        y_train1 = y[train_index]     

        clf_cv1 = RandomForestClassifier(max_depth=opt_hyperparameter1, random_state=1234)

        fselection_cv1 = RFE(clf_cv1, n_features_to_select=n_feat)
        fselection_cv1.fit(x_train1, y_train1)
        x_train1 = fselection_cv1.transform(x_train1)

        clf_cv1.fit(x_train1, y_train1)

        # Test phase
        x_test1 = fselection_cv1.transform(x[test_index, :])
        y_test1 = y[test_index]
        y_pred1 = clf_cv1.predict(x_test1)

        acc_i1 = accuracy_score(y_test1, y_pred1)
        acc_cv1.append(acc_i1)    

    acc1 = np.average(acc_cv1)
    acc_nfeat1.append(acc1)

opt_index1 = np.argmax(acc_nfeat1)
opt_features1 = n_feats[opt_index1]
opt_acc1 = acc_nfeat1[opt_index1]
print("Optimal number of features: ", opt_features1)
print("Its accuracy: ", opt_acc1)

plt.plot(n_feats, acc_nfeat1)
plt.xlabel("Features")
plt.ylabel("Accuracy")

plt.show()

# Fit model with optimal number of features
clf = RandomForestClassifier(max_depth=opt_hyperparameter1, random_state=1234)
fselection1 = RFE(clf, n_features_to_select=n_feat)
fselection1.fit(x, y)

print("Selected features: ", fselection1.get_feature_names_out())

# Combined, applied hyperparameter optimization, feature selection, with Cross-validation

In [ ]:
mm = np.arange(1,20)
acc = []

n_feats = np.arange(1,20)

#Optimización de hiperparámeros 
for m in mm:
    
    acc_hyper=[]

    #Reducción de características
    for n_feat in n_feats:
        
        acc_cv = []

        sss = StratifiedShuffleSplit(
            n_splits=5,
            train_size=0.8,
            random_state=1234
        ) 

        for train_index, test_index in sss.split(x, y):
        
            # Training phase
            x_train = x[train_index, :]
            y_train = y[train_index]     

            clf_cv = RandomForestClassifier(max_depth=m, random_state=1234)

            fselection_cv = 3(clf_cv, n_features_to_select=n_feat)
            fselection_cv.fit(x_train, y_train)
            x_train = fselection_cv.transform(x_train)

            clf_cv.fit(x_train, y_train)

            # Test phase
            x_test = fselection_cv.transform(x[test_index, :])
            y_test = y[test_index]
            y_pred = clf_cv.predict(x_test)

            acc_i = accuracy_score(y_test, y_pred)
            acc_cv.append(acc_i)    

        acc_feats = np.average(acc_cv)
        acc_hyper.append(acc_feats)
    
    acc.append(acc_hyper)

opt_acc=[]
opt_indexes=[]
for item in acc:
    item=np.array(item)
    opt_indexes.append(np.argmax(item))
    opt_acc.append(np.max(item))
opt_hype_index=np.argmax(opt_acc)
opt_feat_index=opt_indexes[opt_hype_index]

opt_features = n_feats[opt_feat_index]
opt_hyperparameter = mm[opt_hype_index]
opt_acc = opt_acc[opt_hype_index]
print("Optimal number of hyperparameters: ", opt_hyperparameter)
print("Optimal number of features: ", opt_features)
print("Its accuracy: ", opt_acc)

"""plt.plot(n_feats, acc_nfeat)
plt.xlabel("Features")
plt.ylabel("Accuracy")

plt.show()"""

# Fit model with optimal number of features
clf = RandomForestClassifier(max_depth=opt_hyperparameter, random_state=1234)
fselection = RFE(clf_cv, n_features_to_select=n_feat)
fselection.fit(x, y)

print("Selected features: ", fselection.get_feature_names_out())